In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Risk_PySpark").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 08:34:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
 # Lire le CSV depuis HDFS
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("nullValue", "N/A") \
    .load("hdfs:///user/mlib/data/dataset_projets_risque.csv")

In [5]:
df.show(5)

+--------------------+------+---------+--------------------+--------------------+----------------+--------------------+--------------------+
|               Titre|Budget|  Secteur|Experience_fondateur|         Description|Montant_collecte|Nom_prenom_fondateur|     Email_fondateur|
+--------------------+------+---------+--------------------+--------------------+----------------+--------------------+--------------------+
|Solution IA marke...| 62464|     Tech|       Intermédiaire|Projet innovant v...|           36007|         Fatma Saidi|fatma.saidi@gmail...|
|     Startup fintech| 96900|  Finance|              Expert|Projet innovant v...|           37920|         Ali Ben Ali|ali.benali@gmail.com|
|Solution IA marke...| 84593|    Santé|              Expert|Projet innovant v...|           13075|      Mohamed Bouzid|mohamed.bouzid@gm...|
|Solution IA marke...| 17002|Éducation|              Expert|Projet innovant v...|            7995|       Yasmine Saidi|yasmine.saidi@gma...|
|Plateforme é

In [7]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# Créer label
from pyspark.sql.functions import col, when

df = df.withColumn(
    "label",
    when(col("Montant_collecte") > col("Budget") * 0.8, 1).otherwise(0)
)

# Features → vecteur
assembler = VectorAssembler(
    inputCols=["Budget", "Montant_collecte"],
    outputCol="features"
)

# Modèle
rf = RandomForestClassifier(featuresCol="features", labelCol="label")

# Pipeline
pipeline = Pipeline(stages=[assembler, rf])

# Train
model = pipeline.fit(df)

# Sauvegarde
model.write().overwrite().save("spark_model")

print("✅ Modèle Spark sauvegardé")

✅ Modèle Spark sauvegardé
